In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!cp "/content/drive/MyDrive/CatsAndDogs.zip" .
!unzip -q CatsAndDogs.zip

In [5]:
!ls

CatsAndDogs.zip  drive	sample_data  train  train.csv  val  val.csv


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory

cudnn.benchmark = True
plt.ion()   # interactive mode

In [6]:
# Data augmentation and normalization for training
# Just normalization for validation
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = '.'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4,
                                             shuffle=True, num_workers=4)
              for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

# We want to be able to train our model on an `accelerator <https://pytorch.org/docs/stable/torch.html#accelerators>`__
# such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [7]:
print(dataset_sizes)
print(class_names)

{'train': 275, 'val': 70}
['cat', 'dog']


In [8]:
model = models.resnet18(weights='IMAGENET1K_V1')  # pretrained weights

# --- Feature extraction mode: freeze everything ---
for param in model.parameters():
    param.requires_grad = False

# --- Now replace the final layer ---
# ResNet18's final layer is model.fc — a Linear layer mapping 512 -> 1000 classes
# New layers you create have requires_grad=True by DEFAULT
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # your new task's class count
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 204MB/s]


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.fc.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)

In [11]:
with TemporaryDirectory() as tempdir:
        best_model_params_path = os.path.join(tempdir, 'best_model_params.pt')

        torch.save(model.state_dict(), best_model_params_path)
        best_acc = 0.0

        for epoch in range(25):
          for phase in ["train","val"]:
            if phase == "train":
              model.train()
            else:
              model.eval()
            running_loss = 0.0
            running_corrects = 0
            for inputs, labels in dataloaders[phase]:
                    inputs = inputs.to(device)
                    labels = labels.to(device)
                    optimizer.zero_grad()
                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)
                        if phase == 'train':
                            loss.backward()
                            optimizer.step()
                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)
            if phase == 'train':
              scheduler.step()
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), best_model_params_path)

        print()
        print(f'Best val Acc: {best_acc:4f}')

        # load best model weights
        model.load_state_dict(torch.load(best_model_params_path, weights_only=True))
model = model

train Loss: 0.6087 Acc: 0.6800
val Loss: 0.4071 Acc: 0.8857
train Loss: 0.5019 Acc: 0.7709
val Loss: 0.3109 Acc: 0.9000
train Loss: 0.3998 Acc: 0.8509
val Loss: 0.2249 Acc: 0.9286
train Loss: 0.4365 Acc: 0.8000
val Loss: 0.1761 Acc: 0.9571
train Loss: 0.4434 Acc: 0.8036
val Loss: 0.1855 Acc: 0.9286
train Loss: 0.4299 Acc: 0.7818
val Loss: 0.1391 Acc: 0.9571
train Loss: 0.3787 Acc: 0.8400
val Loss: 0.1401 Acc: 0.9571
train Loss: 0.3499 Acc: 0.8400
val Loss: 0.1246 Acc: 0.9571
train Loss: 0.3460 Acc: 0.8436
val Loss: 0.1144 Acc: 0.9571
train Loss: 0.3474 Acc: 0.8655
val Loss: 0.1092 Acc: 0.9571
train Loss: 0.3503 Acc: 0.8545
val Loss: 0.1058 Acc: 0.9571
train Loss: 0.3088 Acc: 0.8764
val Loss: 0.1007 Acc: 0.9571
train Loss: 0.3170 Acc: 0.8727
val Loss: 0.1007 Acc: 0.9714
train Loss: 0.2975 Acc: 0.8691
val Loss: 0.1001 Acc: 0.9571
train Loss: 0.3081 Acc: 0.8727
val Loss: 0.1137 Acc: 0.9571
train Loss: 0.3153 Acc: 0.8727
val Loss: 0.0940 Acc: 0.9571
train Loss: 0.2989 Acc: 0.8655
val Loss: